# Notebook 16 — Capstone: MCP + Agents + RAG, One Helpdesk Assistant

> **Local-only notebook.** This notebook spawns a real MCP server as a subprocess over
> stdio and talks to a local SQLite database — both need a local filesystem and process
> spawning, so this will not run in Colab. Run it with Jupyter on your own machine, with
> the `genai2026` kernel, from inside `10_RAG/notebooks/`.

Every notebook up to this point taught one capability at a time: RAG that reads documents
(Notebooks 01–15), tool-calling agents (Module 7), MCP as a standard protocol for tools
(Module 10). This capstone puts all three in one system, doing one real job: a **customer
support helpdesk assistant**.

**The core idea this notebook proves, end to end:** an agent doesn't need to know or care
whether a tool talks to a SQL database or a vector index — MCP makes both look like the
exact same kind of thing, a function with a name, a docstring, and typed arguments. So the
same agent that answers *"what's our refund policy?"* by searching your knowledge base can
also answer *"how many open tickets does Aisha have?"* by joining four database tables — and
neither the agent's code nor the MCP protocol changes between the two.

**What you'll build:**
1. A real 5-table relational database (customers, agents, tickets, ticket_notes, kb_articles)
   with genuine foreign-key relationships — questions that require an actual SQL `JOIN`, not
   just a lookup
2. An MCP server exposing 20 tools across 6 categories: customer CRUD, ticket CRUD, notes,
   cross-table insight tools, and knowledge-base search/write
3. The knowledge-base tool **reuses** the exact production RAG pipeline you already built in
   Notebook 13 (hybrid search + rerank + grounded citations) — imported as a module, not
   rebuilt from scratch
4. A LangChain agent connected to all 20 tools through the MCP protocol
5. A human-in-the-loop guardrail that pauses for approval before any destructive write
   (`close_ticket`, `delete_ticket`) — reusing the `interrupt()` pattern from Module 8

Every piece of code in this notebook was tested standalone before being wired to the next —
tool logic tested directly, then over the real MCP protocol, then through the full agent —
exactly the debugging discipline you should use when you build this yourself.

## 0. Install dependencies

In [20]:
%pip install -q mcp langchain-mcp-adapters "langchain>=1.0.0" langgraph python-dotenv \
    langchain-community langchain-text-splitters langchain-pymupdf4llm pypdf docx2txt \
    sentence-transformers bm25s PyStemmer chromadb openai langchain-openai
print("Dependencies ready.")



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dependencies ready.


## 1. Setup

Same pattern as every notebook since 01. This notebook also needs `nest_asyncio` — Jupyter
already runs an event loop for the kernel itself, and connecting to the MCP server needs its
own nested one.

In [21]:
import warnings, os, sys, asyncio
warnings.filterwarnings("ignore")
import logging
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers", "chromadb", "mcp"):
    logging.getLogger(_n).setLevel(logging.ERROR)

import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd() / "10_RAG" / "notebooks"
CAPSTONE_DIR = NOTEBOOK_DIR / "production_mcp_agents_rag_capstone"
load_dotenv(NOTEBOOK_DIR.parent / ".env")

sys.path.insert(0, str(CAPSTONE_DIR))

print("Setup complete. Capstone folder:", CAPSTONE_DIR)


Setup complete. Capstone folder: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/zero-to-genai-engineer/10_RAG/notebooks/production_mcp_agents_rag_capstone


## 2. The use case: a customer support helpdesk

Five tables, with real foreign-key relationships:

| Table | Purpose | Links to |
|---|---|---|
| `customers` | id, name, email, plan_tier, signup_date | — |
| `agents` | id, name, team | — |
| `tickets` | id, **customer_id**, **agent_id**, subject, description, status, priority | → customers, → agents |
| `ticket_notes` | id, **ticket_id**, author, note_text, created_at | → tickets (one-to-many) |
| `kb_articles` | id, title, content, category | RAG-indexed — this table IS the knowledge base |

A question like *"show every high-priority ticket for Jane Doe, who it's assigned to, and the
latest note"* genuinely requires a 4-table join — this schema isn't decorative, it's what
makes the "insight tools" in Section 4 worth building.

In [22]:
import subprocess
result = subprocess.run([sys.executable, "seed_data.py"], cwd=CAPSTONE_DIR, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


Built helpdesk.db: 22 customers, 6 agents, 25 tickets, 37 notes, 8 KB articles.



## 3. Prove the data is real and joinable — before any MCP or agent is involved

Same discipline as every notebook in this course: verify the ground truth by hand first, so
you know exactly what a correct answer looks like before an LLM is anywhere in the loop.

In [24]:
import sqlite3

def show(sql, params=()):
    conn = sqlite3.connect(CAPSTONE_DIR / "helpdesk.db")
    conn.row_factory = sqlite3.Row
    rows = [dict(r) for r in conn.execute(sql, params).fetchall()]
    conn.close()
    return rows

print("=== Jane Doe's ticket history (a real 3-table JOIN) ===")
for row in show("""
    SELECT t.id, t.subject, t.status, t.priority, a.name AS agent, a.team
    FROM tickets t
    JOIN customers c ON t.customer_id = c.id
    LEFT JOIN agents a ON t.agent_id = a.id
    WHERE c.name = 'Jane Doe'
    ORDER BY t.created_at DESC
"""):
    print(" ", row)

print("\n=== Agent workload (JOIN + GROUP BY) ===")
for row in show("""
    SELECT a.name, a.team, COUNT(t.id) AS open_tickets
    FROM agents a
    LEFT JOIN tickets t ON t.agent_id = a.id AND t.status != 'closed'
    GROUP BY a.id
    ORDER BY open_tickets DESC
"""):
    print(" ", row)


=== Jane Doe's ticket history (a real 3-table JOIN) ===
  {'id': 4, 'subject': 'Integration with our CRM broke', 'status': 'open', 'priority': 'high', 'agent': 'Aisha Rahman', 'team': 'Technical'}
  {'id': 2, 'subject': 'API returning 429 errors', 'status': 'in_progress', 'priority': 'high', 'agent': 'Aisha Rahman', 'team': 'Technical'}
  {'id': 1, 'subject': "Can't reset my password", 'status': 'closed', 'priority': 'medium', 'agent': 'Diego Fernandez', 'team': 'Technical'}
  {'id': 3, 'subject': 'Want to upgrade to Enterprise', 'status': 'closed', 'priority': 'low', 'agent': 'Maria Chen', 'team': 'Billing'}

=== Agent workload (JOIN + GROUP BY) ===
  {'name': 'Aisha Rahman', 'team': 'Technical', 'open_tickets': 4}
  {'name': 'Maria Chen', 'team': 'Billing', 'open_tickets': 2}
  {'name': 'Diego Fernandez', 'team': 'Technical', 'open_tickets': 1}
  {'name': 'Priya Nair', 'team': 'Onboarding', 'open_tickets': 1}
  {'name': 'Tom Okafor', 'team': 'Billing', 'open_tickets': 0}
  {'name': "

## 4. The MCP server — 20 tools across 6 categories

The full server lives in `production_mcp_agents_rag_capstone/mcp_server.py` — one Python file,
imported and reused, not something this notebook regenerates inline. Every tool is a plain
function: a `@mcp.tool()` decorator, a docstring the LLM reads to decide when to call it, and
type-hinted arguments FastMCP turns into a JSON schema automatically.

```python
@mcp.tool()
def create_ticket(customer_id: int, subject: str, description: str,
                   priority: str = "medium") -> dict:
    """Open a new support ticket for a customer. priority must be one of: low, medium, high.
    The ticket starts unassigned (no agent) and with status 'open'. Returns the new ticket."""
    ...

@mcp.tool()
def close_ticket(ticket_id: int, resolution_note: str) -> dict:
    """Close a ticket AND record how it was resolved, in one step. This is a
    DESTRUCTIVE, hard-to-reverse action — always confirm with the user what the
    resolution was before calling this."""
    ...
```

The six categories: **customer tools** (create/get/update/search), **agent tools** (list),
**ticket tools** (create/get/update status/update priority/assign/close/delete/list-by-customer/
list-by-agent), **notes tools** (add/get), **cross-table insight tools** (Section 5), and
**knowledge tools** (Section 6) — 20 tools total.

Before wiring anything to an agent or even the MCP protocol, call the tool functions directly
— they're plain Python functions underneath the decorator, so this proves the server's actual
logic is correct with zero extra moving parts.

In [ ]:
import mcp_server as tools

new_ticket = tools.create_ticket(1, "Dashboard won't load", "Blank screen after login.", priority="high")
print("Created:", new_ticket)

fetched = tools.get_ticket(new_ticket["id"])
print("\nFetched back:", fetched)

assigned = tools.assign_ticket(new_ticket["id"], 3)  # Aisha Rahman
print("\nAssigned to agent 3:", assigned["agent_id"])

bad = tools.create_customer("Test", "test@example.com", "gold")
print("\nInvalid plan_tier correctly rejected:", bad)


In [5]:
import mcp_server as tools

new_ticket = tools.create_ticket(1, "Dashboard won't load", "Blank screen after login.", priority="high")
print("Created:", new_ticket)

fetched = tools.get_ticket(new_ticket["id"])
print("\nFetched back:", fetched)

assigned = tools.assign_ticket(new_ticket["id"], 3)  # Aisha Rahman
print("\nAssigned to agent 3:", assigned["agent_id"])

bad = tools.create_customer("Test", "test@example.com", "gold")
print("\nInvalid plan_tier correctly rejected:", bad)


Created: {'id': 26, 'customer_id': 1, 'agent_id': None, 'subject': "Dashboard won't load", 'description': 'Blank screen after login.', 'status': 'open', 'priority': 'high', 'screenshot_path': None, 'created_at': '2026-08-08 20:26:31', 'updated_at': '2026-08-08 20:26:31'}

Fetched back: {'id': 26, 'customer_id': 1, 'agent_id': None, 'subject': "Dashboard won't load", 'description': 'Blank screen after login.', 'status': 'open', 'priority': 'high', 'screenshot_path': None, 'created_at': '2026-08-08 20:26:31', 'updated_at': '2026-08-08 20:26:31', 'customer_name': 'Jane Doe', 'agent_name': None}

Assigned to agent 3: 3

Invalid plan_tier correctly rejected: {'error': "Invalid plan_tier 'gold' — must be free, pro, or enterprise."}


## 5. The JOIN-powered insight tools — the whole point of this schema

These two tools don't just SELECT from one table — they JOIN across customers, tickets,
agents, and ticket_notes, and return one combined answer instead of four separate lookups an
agent would otherwise have to stitch together itself.

In [6]:
import json

jane = tools.search_customers("Jane Doe")[0]
history = tools.get_customer_support_history(jane["id"])
print(f"{history['customer']['name']}: {history['total_tickets']} total tickets, "
      f"{history['open_tickets']} still open\n")
for t in history["tickets"]:
    print(f"  [{t['status']:11}] {t['subject']}  (agent: {t['agent_name']})")

print("\n" + "=" * 60)
print("Agent workload right now:")
for row in tools.get_agent_workload():
    print(f"  {row['name']:18} {row['team']:11} active={row['active_tickets']}  closed={row['closed_tickets']}")


Jane Doe: 5 total tickets, 3 still open

  [open       ] Dashboard won't load  (agent: Aisha Rahman)
  [open       ] Integration with our CRM broke  (agent: Aisha Rahman)
  [in_progress] API returning 429 errors  (agent: Aisha Rahman)
  [closed     ] Can't reset my password  (agent: Diego Fernandez)
  [closed     ] Want to upgrade to Enterprise  (agent: Maria Chen)

Agent workload right now:
  Aisha Rahman       Technical   active=5  closed=3
  Maria Chen         Billing     active=2  closed=3
  Diego Fernandez    Technical   active=1  closed=5
  Priya Nair         Onboarding  active=1  closed=0
  Tom Okafor         Billing     active=0  closed=3
  Liam O'Brien       Onboarding  active=0  closed=1


## 6. The knowledge-base tool — reusing Notebook 13's RAG pipeline, not rebuilding it

This is the key architectural decision in this notebook: `search_knowledge_base` doesn't
reimplement hybrid search and reranking — it **imports `ProductionRAGChatbot` from
`production_rag_chatbot/rag_pipeline.py`** and calls it. Real production systems package RAG
as a reusable service and call it from wherever they need it; they don't re-derive the
pipeline every time a new feature needs search.

```python
sys.path.insert(0, str(HERE.parent / "production_rag_chatbot"))
from rag_pipeline import ProductionRAGChatbot

def _get_rag_bot():
    global _rag_bot
    if _rag_bot is None:
        _rag_bot = ProductionRAGChatbot(top_k=8, top_n=4, model="gpt-4o-mini")
        _rag_bot.ingest(sorted(str(p) for p in KB_DIR.glob("*.txt")))
    return _rag_bot

@mcp.tool()
def search_knowledge_base(question: str) -> dict:
    """Answer a question from the support knowledge base ... Use this for
    'how do I...' / 'what is your policy on...' style questions — NOT for looking up
    a specific customer's ticket, which lives in the database instead."""
    bot = _get_rag_bot()
    bot.history = []  # each tool call is independent, no cross-call memory bleed
    result = bot.chat(question)
    return {"answer": result["answer"], "sources": [...]}
```

The 8 knowledge base articles (password reset, billing FAQ, rate limits, 2FA, ...) started
life as rows in the `kb_articles` table — the DB is the source of truth, and they're
materialized as `.txt` files purely because the RAG pipeline's `ingest()` takes file paths.

In [7]:
result = tools.search_knowledge_base("How long is my password reset link valid for?")
print("Answer:", result["answer"])
print("Sources:", result["sources"])

result2 = tools.search_knowledge_base("What is the API rate limit on the Pro tier?")
print("\nAnswer:", result2["answer"])


[08/08/26 20:26:32] WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please   ]8;id=234053;file:///opt/homebrew/lib/python3.11/site-packages/huggingface_hub/utils/_http.py\_http.py]8;;\:]8;id=146316;file:///opt/homebrew/lib/python3.11/site-packages/huggingface_hub/utils/_http.py#916\916]8;;\
                             set a HF_TOKEN to enable higher rate limits and faster downloads.                     

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[08/08/26 20:26:38] DEBUG    Building index from tokens                                             ]8;id=91161;file:///opt/homebrew/lib/python3.11/site-packages/bm25s/__init__.py\__init__.py]8;;\:]8;id=619176;file:///opt/homebrew/lib/python3.11/site-packages/bm25s/__init__.py#517\517]8;;\

Answer: The password reset link is valid for 30 minutes. If it expires, you can request a new one from the same page [1].
Sources: [{'title': '01_how_to_reset_your_password.txt', 'score': 9.608}, {'title': '07_troubleshooting_failed_integrations.txt', 'score': -3.618}, {'title': '01_how_to_reset_your_password.txt', 'score': -4.688}, {'title': '08_our_cancellation_and_refund_policy.txt', 'score': -6.772}]

Answer: The API rate limit on the Pro tier allows for 1,000 requests per minute [1].


### The write-back loop — a resolved ticket becomes a new searchable article

`create_kb_article` inserts into the DB, re-exports the `.txt` files, and forces the RAG index
to rebuild on the next search — so a new article is searchable immediately, not after some
separate reindexing job.

In [8]:
new_article = tools.create_kb_article(
    "How to Rotate an Expired API Key",
    "If your API key expired after 90 days of inactivity, generate a new one from "
    "Settings > API Keys > Rotate Key. The old key remains valid for 24 hours to allow "
    "a graceful cutover.",
    "Technical",
)
print("Published:", new_article["title"])

result3 = tools.search_knowledge_base("How long does the old API key stay valid after I rotate it?")
print("\nImmediately searchable:", result3["answer"])


Published: How to Rotate an Expired API Key


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[08/08/26 20:26:47] DEBUG    Building index from tokens                                             ]8;id=229258;file:///opt/homebrew/lib/python3.11/site-packages/bm25s/__init__.py\__init__.py]8;;\:]8;id=243962;file:///opt/homebrew/lib/python3.11/site-packages/bm25s/__init__.py#517\517]8;;\


Immediately searchable: The old API key remains valid for 24 hours after you rotate it to allow for a graceful cutover to the new key [1].


## 7. Starting the real MCP server and connecting LangChain to it

Everything above called the tool *functions* directly — useful for testing, but it skips the
actual MCP protocol. Now start `mcp_server.py` as a real subprocess (stdio transport) and
connect to it the way a production client would: `MultiServerMCPClient` speaks the protocol,
and `get_tools()` turns every discovered MCP tool into a normal LangChain `Tool` — the exact
same interface as a plain Python function tool from Module 7, whether it's backed by SQL or
by the full RAG pipeline underneath.

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "helpdesk": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(CAPSTONE_DIR / "mcp_server.py")],
    }
})
mcp_tools = await client.get_tools()
print(f"Discovered {len(mcp_tools)} tools over the real MCP protocol:\n")
for t in mcp_tools:
    print(" -", t.name)


Discovered 20 tools over the real MCP protocol:

 - create_customer
 - get_customer
 - update_customer
 - search_customers
 - list_agents
 - create_ticket
 - get_ticket
 - update_ticket_status
 - update_ticket_priority
 - assign_ticket
 - close_ticket
 - delete_ticket
 - list_tickets_by_customer
 - list_tickets_by_agent
 - add_ticket_note
 - get_ticket_notes
 - get_customer_support_history
 - get_agent_workload
 - search_knowledge_base
 - create_kb_article


## 8. Building the agent, with a human-in-the-loop guardrail

`HumanInTheLoopMiddleware` pauses the agent — using the same `interrupt()` mechanism from
Module 8 — before it's allowed to actually run `close_ticket` or `delete_ticket`. Every other
tool (reads, creates, notes, KB search) executes immediately with no pause. This is a
deliberate, configurable decision per tool, not a blanket "ask before every action" — that
would make the agent useless to actually talk to.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

SYSTEM_PROMPT = """You are a helpful customer support desk assistant. You have tools to:
- look up and manage customers, tickets, agents, and ticket notes in our database
- search our knowledge base for policy/how-to questions (search_knowledge_base)

Always use a tool to look up real data rather than guessing. When a user gives a
customer's name instead of an id, use search_customers first to find their id.
Be concise and factual in your answers."""

agent = create_agent(
    model="gpt-4o-mini",
    tools=mcp_tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"close_ticket": True, "delete_ticket": True})],
    checkpointer=InMemorySaver(),
)
print("Agent ready — 20 tools, HITL guardrail on close_ticket + delete_ticket.")


Agent ready — 20 tools, HITL guardrail on close_ticket + delete_ticket.


## 9. Test drive — every tool category, one conversation at a time

Nine real scenarios, each checked against the same ground truth verified in Section 3 —
single-table reads, multi-table joins, knowledge-base search, writes, and both the approve and
reject paths of the destructive-write guardrail.

In [11]:
async def ask(question, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]}, config=config)
    return result, config

result, _ = await ask("What plan tier is the customer Jane Doe on?", "demo-1")
print("Q: What plan tier is the customer Jane Doe on?")
print("A:", result["messages"][-1].content)


Q: What plan tier is the customer Jane Doe on?
A: Jane Doe is on the "enterprise" plan tier.


In [12]:
result, _ = await ask(
    "Give me Jane Doe's full support history — how many tickets has she opened, "
    "and how many are still open?", "demo-2")
print("Q: Give me Jane Doe's full support history...")
print("A:", result["messages"][-1].content)


Q: Give me Jane Doe's full support history...
A: Jane Doe has opened a total of **5 tickets**, with **3 tickets still open**. Here is a summary of her support history:

1. **Open Tickets:**
   - **Subject:** Dashboard won't load
     - **Status:** Open
     - **Priority:** High
     - **Created At:** August 8, 2026
     - **Agent:** Aisha Rahman (Technical)
  
   - **Subject:** Integration with our CRM broke
     - **Status:** Open
     - **Priority:** High
     - **Created At:** July 31, 2026
     - **Agent:** Aisha Rahman (Technical)
  
   - **Subject:** API returning 429 errors
     - **Status:** In Progress
     - **Priority:** High
     - **Created At:** July 27, 2026
     - **Agent:** Aisha Rahman (Technical)
     - **Latest Note:** Applied the fix, monitoring for recurrence (July 2, 2026).

2. **Closed Tickets:**
   - **Subject:** Can't reset my password
     - **Status:** Closed
     - **Priority:** Medium
     - **Created At:** June 22, 2026
     - **Agent:** Diego Fernandez (

In [13]:
result, _ = await ask("Which support agent currently has the most active tickets?", "demo-3")
print("Q: Which support agent currently has the most active tickets?")
print("A:", result["messages"][-1].content)


Q: Which support agent currently has the most active tickets?
A: The support agent with the most active tickets is **Aisha Rahman** from the Technical team, who currently has **5 active tickets**.


In [14]:
result, _ = await ask(
    "A customer is asking how many two-factor authentication backup codes they get.", "demo-4")
print("Q: How many 2FA backup codes does a customer get?")
print("A:", result["messages"][-1].content)


Q: How many 2FA backup codes does a customer get?
A: Customers receive 10 single-use backup codes when they enable two-factor authentication.


In [15]:
result, _ = await ask(
    "Open a new high priority ticket for Jane Doe: subject 'Cannot log in', "
    "description 'Getting a 500 error on the login page.'", "demo-5")
print("Q: Open a new high priority ticket for Jane Doe...")
print("A:", result["messages"][-1].content)


Q: Open a new high priority ticket for Jane Doe...
A: A high priority ticket has been successfully opened for Jane Doe with the following details:

- **Ticket ID**: 27
- **Subject**: Cannot log in
- **Description**: Getting a 500 error on the login page.
- **Status**: Open

If you need further assistance or updates, feel free to ask!


### The guardrail in action — approve path

Watch the agent pause instead of just closing the ticket.

In [16]:
result, config = await ask(
    "Please close ticket 3 with resolution note: 'Resolved via phone call — walked customer through the fix.'",
    "demo-6")
print("Interrupted, awaiting approval?", "__interrupt__" in result)
if "__interrupt__" in result:
    print(result["__interrupt__"])


Interrupted, awaiting approval? True
[Interrupt(value={'action_requests': [{'name': 'close_ticket', 'args': {'ticket_id': 3, 'resolution_note': 'Resolved via phone call — walked customer through the fix.'}, 'description': "Tool execution requires approval\n\nTool: close_ticket\nArgs: {'ticket_id': 3, 'resolution_note': 'Resolved via phone call — walked customer through the fix.'}"}], 'review_configs': [{'action_name': 'close_ticket', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='c2d61955cd426ee2a711005d435d8829')]


In [17]:
# Approve — the tool actually runs now.
result2 = await agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print("After approval:", result2["messages"][-1].content)


After approval: The ticket has been successfully closed with the resolution note: "Resolved via phone call — walked customer through the fix." If you need further assistance, feel free to ask!


### The guardrail in action — reject path

Same flow, but this time we reject it. The ticket must stay untouched.

In [18]:
result3, config8 = await ask(
    "Please close ticket 4 with resolution note: 'test reject.'", "demo-7")
print("Interrupted again?", "__interrupt__" in result3)

result4 = await agent.ainvoke(Command(resume={"decisions": [{"type": "reject"}]}), config=config8)
print("After rejection:", result4["messages"][-1].content)

# Prove it at the database level, not just from the LLM's wording.
row = show("SELECT id, status FROM tickets WHERE id = 4")
print("\nTicket 4 status in the DB after rejection (should NOT be 'closed'):", row)


Interrupted again? True
After rejection: It seems there was an issue while trying to close ticket 4. Can you please confirm the resolution note again or provide any additional details?

Ticket 4 status in the DB after rejection (should NOT be 'closed'): [{'id': 4, 'status': 'open'}]


### The full loop — knowledge base write-back, through the agent this time

In [19]:
result, _ = await ask(
    "Please publish a new knowledge base article titled 'How to Change Your Billing Email' "
    "in the Billing category. Content: 'Go to Settings > Billing > Update Email. Changes take "
    "effect on your next invoice, not retroactively.' After publishing, search the knowledge base "
    "to confirm: when do billing email changes take effect?",
    "demo-8")
print("A:", result["messages"][-1].content)


A: The article titled **'How to Change Your Billing Email'** has been successfully published in the **Billing** category. 

Regarding your query, billing email changes take effect on your next invoice, not retroactively.


## 10. The full picture

One MCP server. Twenty tools. Structured SQL tools and a full hybrid-search RAG pipeline,
reachable through the exact same protocol, called by the exact same agent, with the exact
same `@mcp.tool()` decorator. The agent never had to know which kind of tool it was calling —
that's the entire point of building this on MCP instead of hand-wiring each tool type
differently.

## 11. What's still missing — setting up the next class

- **Concurrency**: this notebook runs one conversation at a time. A real helpdesk serves many
  customers simultaneously — the MCP server would need connection pooling and the SQLite file
  would need to become a real Postgres database.
- **Auth**: any user of this agent can currently look up any customer's data. Production needs
  per-user scoping — an agent acting on behalf of Customer X should never be able to fetch
  Customer Y's tickets.
- **Streaming**: the test-drive calls above wait for the full answer. A real chat UI streams
  tokens as they're generated (Module 8 covered this — the same pattern applies here).
- **Observability**: which tool got called, how long it took, whether the RAG tool's retrieval
  was actually good that day — none of that is logged yet. That's Module 16 (LLMOps &
  Evaluation).

## 12. Summary

**What you built:** a real 5-table relational database, a 20-tool MCP server spanning SQL
CRUD, cross-table JOIN insight tools, and a reused production RAG pipeline exposed as one more
tool — all connected to one LangChain agent, with a human-in-the-loop guardrail proven to
actually block a destructive write at the database level, not just in the conversation text.

**The idea that generalizes:** MCP's real value isn't "a way to call tools" — LangChain could
already do that. It's that a tool's *implementation* (SQL query, vector search, a JOIN across
four tables, an API call) becomes invisible to the agent. The agent just sees a name, a
docstring, and a schema. That's what let this notebook add a completely different kind of
tool — grounded RAG search — to the same agent, the same server, the same protocol, with zero
special-casing anywhere in the agent's own code.